<div style="text-align: justify; max-width: 90ch">

# Lecture 01b: Choosing a model, requirements first

**Status: Mandatory reading.**

In `lec_01a` you ran `qwen3:1.7b` because the course chose it for you. How was it chosen? Not by picking the biggest model that fits a laptop.
A model is a component of a system that has a job, and the job comes with **requirements**: what it must do, in which languages, under which license, at what speed, on which hardware, with what privacy.
Size and hardware fit come *after* that list, never before it.

**What this notebook covers**

- The **requirements table**: do this before looking at any model.
- The **Hugging Face Hub** from Python; **model cards**, and what "open weights" allows.
- **Quantization**: why one model comes in twenty file sizes.
- Hardware fit with `llmfit`.
- Two candidates compared on the same prompts, and what thinking costs.
- A short **decision record**, a skill that you should develop.

</div>

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path

import ollama
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import HfApi, ModelCard

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. The wrong question and the right one

"Which model is best?" has no answer. "Which model is best *for this job*?" does, once the job is written down. Choosing by size or by popularity fails in several ways:

- **A license that forbids the product.** Some downloadable models may not be used commercially.
- **A language the model handles badly.** It still answers, so nothing warns you.
- **Hardware the users do not have**, or a hosted model that sends private documents to someone else's servers.

A **requirement** is a condition the model must meet for the system to do its job. Here is the requirements table for this course's own system:

> An assistant that answers questions about the Python course notes on a student laptop.

</div>

<div style="text-align: justify; max-width: 90ch">

| Requirement | For this course | Why it matters |
| --- | --- | --- |
| **Task** | answer questions from retrieved notes (lecture 2); call tools (lecture 3) | not every small model follows instructions or calls tools |
| **Languages** | English notes; Greek questions must work | the tokenizer and training data decide this |
| **License** | free for any use, including commercial, no naming clauses | a product built on it must be shippable |
| **Context** | at least 8 000 tokens (notes + question + answer) | lecture 2 puts several pages into one prompt |
| **Latency** | interactive on a CPU laptop: 10+ tokens/s | a 60-second answer is not a chat |
| **Hardware and cost** | 8-16 GB RAM, no GPU, no API bill | who the students are |
| **Privacy** | everything stays on the laptop | course notes are harmless; a company's documents are not |
| **Maintenance** | one active family with several sizes | the `cpu` and `gpu` tiers should share one code base |

</div>

<div style="text-align: justify; max-width: 90ch">

```text
WRONG: "the biggest model that fits in my RAM"
RIGHT: "the smallest model that meets every requirement, then the biggest of those that fits"
```

The right rule has two steps:

- The requirements decide which models qualify. The smallest that qualifies is the floor, the model every student's laptop can run:
  the `cpu` tier.
- The hardware then decides how far above it you may go: the largest qualifying model that fits your RAM, or a `gpu`-tier one.

In the table, **latency** is how long the user waits for an answer, set by tokens per second (`lec_01a`); **tool calling** means the model asks your code to run a function and uses the result (lecture 3).

The rest of this notebook checks candidates against that requirements table.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Where models live: the Hugging Face Hub

A published model is a set of files:

- the weights
- the tokenizer (`lec_01a`)
- a configuration

To use one you must:

- **find** it among many models;
- **download** exactly the version you tested;
- **know** what you may do with it.

The [Hugging Face Hub](https://huggingface.co/docs/hub/index) (`lec_01a`) solves all three problems: it is to models what GitHub is to code, with over two million models, plus datasets and apps (**Spaces**;
you hosted a Gradio app on one in the Python course).

| On the Hub | What you are familiar with |
| --- | --- |
| a **repository**, such as `Qwen/Qwen3-1.7B` | a GitHub repository: files with a git history |
| its **repo id**, `<owner>/<name>` | GitHub's `owner/name`; the owner is a user or an **organisation** (`Qwen`, `google`) |
| the **model card** | the repository's `README.md` (section 3) |
| the `huggingface_hub` library | the `ollama` package: a Python client for a service (`lec_01a`) |

</div>

<div style="text-align: justify; max-width: 90ch">

The repo id is how every tool names a model.

The `huggingface_hub` library does what a browser cannot: one call filters thousands of models, the results arrive as Python objects ready for a DataFrame,
and the same cell repeats the search next month.

Public information needs no login, but anonymous requests are [rate-limited](https://huggingface.co/docs/hub/rate-limits),
and a **gated** model serves its files only to accounts that accepted its license.

Guide `01d_env_hugging_face` shows how to create a HF **read token** and put it in `.env`.

The configuration cell already copied `.env` into environment variables (`lec_01a`, section 0), and `huggingface_hub` reads `HF_TOKEN` from there.
The token should not appear in the notebook, and it should never end up on GitHub.

</div>

In [3]:
token = os.environ.get("HF_TOKEN")
print("HF_TOKEN loaded:", bool(token), "" if not token else f"({token[:6]}...)")

HF_TOKEN loaded: False 


<div style="text-align: justify; max-width: 90ch">

The saved output says `False`: that run had no token, and every cell below still worked, because they read only public information.
Even a gated model's card is public; only the other files are locked. The token matters when you download often, or download a gated model.

</div>

<div style="text-align: justify; max-width: 90ch">

Start from the **languages** requirement. `HfApi()` creates the client that sends requests to the Hub, like `ollama.Client` in `lec_01a`,
and its `list_models` searches the **tags** the Hub attaches to every model ([search guide](https://huggingface.co/docs/huggingface_hub/guides/search)):

- `filter="el"`: the language tag for Greek, a two-letter code.
- `pipeline_tag="text-generation"`: the task tag; this one keeps models that write text.
- `sort="downloads"`: the most downloaded first; `limit=10` keeps ten.

</div>

In [4]:
api = HfApi()

greek_models = api.list_models(
    pipeline_tag="text-generation", filter="el", sort="downloads", limit=10
)
pd.DataFrame(
    [{"model": m.id, "downloads": m.downloads, "likes": m.likes} for m in greek_models]
)

,model,downloads,likes
0,utter-project/EuroLLM-22B-Instruct-2512,355242,85
1,utter-project/EuroLLM-1.7B-Instruct,216808,104
2,speakleash/Bielik-11B-v3.0-Instruct-awq,81767,3
3,facebook/xglm-564M,75805,54
4,BSC-LT/salamandra-7b-instruct,26361,84
5,utter-project/EuroLLM-9B-Instruct-2512,26237,16
6,CohereLabs/aya-expanse-8b,19135,452
7,utter-project/EuroLLM-1.7B,16636,120
8,utter-project/EuroLLM-9B-Instruct,15438,223
9,bartowski/c4ai-command-r7b-12-2024-GGUF,13621,15


<div style="text-align: justify; max-width: 90ch">

Read the names: most follow a pattern.

| Part | Example | Meaning |
| --- | --- | --- |
| **owner** | `utter-project/` | who published it |
| **family** | `EuroLLM`, `salamandra` | one publisher's series of models, in several sizes |
| **size** | `9B`, `7b` | billions of parameters (`lec_01a`) |
| **variant** | `Instruct`, `it` | **fine-tuned** (trained further) on examples of instructions and answers, so it follows instructions, which chat needs |
| **format** | `awq`, `GGUF` | compressed weights (section 4) |

Without that fine-tuning, a **base** model only continues the text you give it.

</div>

<div style="text-align: justify; max-width: 90ch">

Popular is not the same as suitable. `downloads` counts the last 30 days and `likes` work like GitHub stars: both measure attention, not fitness for your job.
In the saved run, `facebook/xglm-564M` ranks third: a research model with no instruction tuning, built to continue text, not to chat.
Names are a convention, not a rule: `Qwen/Qwen3-1.7B` has no `Instruct` and is tuned for chat all the same. Section 3 shows where to check.

Search results change every month; this list will look different when you run it. The requirements table does not change, and that is the point of writing it first.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Reading a model card

A **model card** is the `README.md` of a model repository. The idea comes from a [2018 paper](https://arxiv.org/abs/1810.03993):
a released model should come with a short document saying what it is meant for and how well it works, so that nobody uses it where it does not fit.
On the Hub a card has two parts ([docs](https://huggingface.co/docs/hub/model-cards)):

- **Metadata**: a header of fixed fields (`license`, `language`, `base_model`, `pipeline_tag`) that the Hub turns into search filters;
  section 2 searched it.
- **Text**: the publisher's own prose about intended use, training, limitations and recommended settings.

Three candidates from three companies, side by side. `ModelCard.load` fetches only the README, not the weights.

</div>

In [5]:
candidates = [
    "Qwen/Qwen3-1.7B",
    "meta-llama/Llama-3.2-1B-Instruct",
    "google/gemma-3-1b-it",
]

rows = []
for repo in candidates:
    data = ModelCard.load(repo).data.to_dict()
    rows.append(
        {
            "model": repo,
            "license": data.get("license"),
            "languages (metadata)": data.get("language"),
            "base_model": data.get("base_model"),
        }
    )
pd.DataFrame(rows)

,model,license,languages (metadata),base_model
0,Qwen/Qwen3-1.7B,apache-2.0,None,[Qwen/Qwen3-1.7B-Base]
1,meta-llama/Llama-3.2-1B-Instruct,llama3.2,"[en, de, fr, it, pt, hi, es, th]",None
2,google/gemma-3-1b-it,gemma,None,google/gemma-3-1b-pt


<div style="text-align: justify; max-width: 90ch">

Each column answers a row of the requirements table:

| Field | Requirement row | What it holds |
| --- | --- | --- |
| `license` | License | an identifier from the Hub's [license list](https://huggingface.co/docs/hub/repositories-licenses), which also names custom agreements such as `llama3.2` and `gemma` |
| `language` | Languages | two-letter codes, as the publisher declared them |
| `base_model` | Task | the model it was fine-tuned from: `Qwen3-1.7B-Base` for Qwen3, `gemma-3-1b-pt` (pre-trained) for Gemma's `it` |

`None` means the publisher left the field empty, not that the answer is no.

</div>

<div style="text-align: justify; max-width: 90ch">

Against the requirements table:

- [**Llama 3.2**](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) lists eight languages and Greek is not one of them:
  it fails the *languages* requirement before we look at anything else. Its license, `llama3.2`, is a custom agreement with conditions on naming and on very large services:
  a legal review, not a checkbox.
- **Gemma 3** ships under Google's own [`gemma` terms](https://ai.google.dev/gemma/terms), also custom.
- **Qwen3** is [`apache-2.0`](https://www.apache.org/licenses/LICENSE-2.0), a **permissive** license: any use, including commercial, as long as you keep the license notices.
  MLflow uses it too; pandas and scikit-learn use BSD, a similar one. Its metadata says nothing about languages.

When metadata is missing, read the card text.

</div>

<div style="text-align: justify; max-width: 90ch">

All three are **open weights**: the weight files are public downloads, for a gated model once your access request is granted.
That answers neither of two other questions:

- **What may you do with them?** The license decides. `CohereLabs/tiny-aya-base`, in section 2's saved run, calls itself an "open weights research release" and is licensed `cc-by-nc-4.0`:
  [non-commercial](https://creativecommons.org/licenses/by-nc/4.0/). Llama's license asks for "Built with Llama" on your product and a separate license above 700 million monthly users;
  Gemma's terms make you pass Google's [prohibited-use policy](https://ai.google.dev/gemma/prohibited_use_policy) on to your own users.
- **Is it open source?** The [Open Source AI Definition](https://opensource.org/ai/open-source-ai-definition) also requires the training code and enough information about the training data to rebuild the model;
  open weights alone do not meet it.

</div>

In [ ]:
qwen_card = ModelCard.load("Qwen/Qwen3-1.7B")

claims = [
    line.strip() for line in qwen_card.text.splitlines() if "languages" in line.lower()
]

for claim in claims[:3]:
    print(claim[:160])

- **Support of 100+ languages and dialects** with strong capabilities for **multilingual instruction following** and **translation**.


<div style="text-align: justify; max-width: 90ch">

"Support of 100+ languages" is a claim by the vendor, not a measurement. Section 6 tests it on a Greek question, because a requirement is met when *you* have checked it.

The missing metadata also explains section 2: Qwen3 declares no language tags, so `filter="el"` could never have found it.
A search finds what publishers declared, not what models can do.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Size and quantization

The names in section 2 carried a size, such as `9B`: billions of parameters (`lec_01a`). Size drives two requirements:

- **Memory** is arithmetic: **parameters × bytes per parameter**. Models are usually published in 16-bit numbers, `BF16` or `FP16`, which take 2 bytes per weight:
  the original [`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B) files hold 2.0 billion numbers (`lec_01a` explained why not 1.7),
  4.1 GB.
- **Speed** follows memory. To generate each token the hardware reads every weight once, so a smaller file also generates faster.

**Quantization** rounds the weights to fewer bits after training: the file shrinks and runs faster, and the model gets a little worse, because each weight is stored less exactly.

</div>

<div style="text-align: justify; max-width: 90ch">

| Format | Bytes per weight | File in the repository below | Typical quality loss |
| --- | --- | --- | --- |
| `BF16` / `FP16` | 2 | 3.4 GB | none (full precision) |
| `Q8_0` | 1 | 1.8 GB | negligible |
| `Q4_K_M` | about 0.6 | 1.1 GB | small; the usual default |
| `Q2_K` | under 0.5 | 0.8 GB | noticeable |

**GGUF** is the file format that Ollama runs, through [`llama.cpp`](https://github.com/ggml-org/llama.cpp), the C++ **engine** that does the model's arithmetic.
One GGUF file holds the quantized weights and what is needed to run them, such as the tokenizer ([Hub docs](https://huggingface.co/docs/hub/gguf)).
Community accounts publish the same model in every quantization; `model_info` with `files_metadata=True` lists a repository's files, its `siblings`, with their sizes.

</div>

In [ ]:
gguf_repo = "unsloth/Qwen3-1.7B-GGUF"
info = api.model_info(gguf_repo, files_metadata=True)

gguf_files = pd.DataFrame(
    [
        {"file": s.rfilename, "size_gb": round(s.size / 1e9, 2)}
        for s in info.siblings
        if s.rfilename.endswith(".gguf")
    ]
)

gguf_files["quantization"] = gguf_files["file"].str.extract(r"Qwen3-1\.7B-(.+)\.gguf")
gguf_files.sort_values("size_gb")

,file,size_gb,quantization
16,Qwen3-1.7B-UD-IQ1_S.gguf,0.54,UD-IQ1_S
15,Qwen3-1.7B-UD-IQ1_M.gguf,0.56,UD-IQ1_M
18,Qwen3-1.7B-UD-IQ2_XXS.gguf,0.61,UD-IQ2_XXS
17,Qwen3-1.7B-UD-IQ2_M.gguf,0.71,UD-IQ2_M
19,Qwen3-1.7B-UD-IQ3_XXS.gguf,0.77,UD-IQ3_XXS
3,Qwen3-1.7B-Q2_K.gguf,0.78,Q2_K
4,Qwen3-1.7B-Q2_K_L.gguf,0.78,Q2_K_L
20,Qwen3-1.7B-UD-Q2_K_XL.gguf,0.80,UD-Q2_K_XL
6,Qwen3-1.7B-Q3_K_S.gguf,0.87,Q3_K_S
5,Qwen3-1.7B-Q3_K_M.gguf,0.94,Q3_K_M


<div style="text-align: justify; max-width: 90ch">

The names encode the method. At the level you need:

| Part | Example | Meaning |
| --- | --- | --- |
| `Q` and a number | `Q4_K_M` | about that many bits per weight |
| `K` | `Q4_K_M` | llama.cpp's **k-quants**: weights in small blocks, each with its own scale, so "4-bit" costs about 4.5 bits |
| `S`, `M`, `L` | `Q4_K_M` | a small, medium or large **mix**: the larger the mix, the more parts of the model keep extra bits |
| `_0`, `_1` | `Q8_0` | older, simpler methods |
| `IQ` | `IQ4_XS` | **i-quants**: an importance measurement, taken on sample text, decides where the bits go |
| `UD-` | `UD-Q4_K_XL` | "Unsloth Dynamic", this account's own mixes ([card](https://huggingface.co/unsloth/Qwen3-1.7B-GGUF)) |

Details: [llama.cpp's quantize tool](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md).

</div>

<div style="text-align: justify; max-width: 90ch">

The `Q4_K_M` file is a third of the `BF16` one. A label names a recipe, not an exact file:

- **Mixed bits.** `Q4_K_M` is mostly 4-bit: in Ollama's `qwen3:1.7b`, 73% of the weights are 4-bit and most of the rest 6-bit.
- **Packaging.** This repository stores the shared vocabulary table (`lec_01a`) once, so its `BF16` file is 3.4 GB against the original's 4.1 GB.
  Ollama's build stores it twice, like the original, so its `Q4_K_M` is 1.4 GB (`lec_01a`), not 1.1.

Same weights, same label, different files: compare sizes within one publisher's repository.

</div>

<div style="text-align: justify; max-width: 90ch">

Weights are not the whole bill: the context window (`lec_01a`) needs memory for every token it holds. In one example run, `qwen3:1.7b` used 1.9 GB loaded with Ollama's default 4096-token window and 2.4 GB with 8192.
That total is what the RAM, or the GPU, must hold.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Does it fit? Ask llmfit

The requirements table asks whether a model *should* do the job; the hardware decides whether it *can*, fast enough. [`llmfit`](https://github.com/AlexsJones/llmfit) (guide `01b_llmfit`) answers without downloading any model:

- **It measures** your CPU, RAM, GPU and VRAM (`lec_01a`): a model runs at GPU speed only if it fits in VRAM.
- **It estimates** each model in its catalog with section 4's arithmetic: memory from parameters and quantization, speed from how fast the memory can be read ([how it works](https://github.com/AlexsJones/llmfit/blob/main/docs/how-it-works.md)).
- **It scores** each model on **fit**, **speed**, **quality** and **context**.

In a terminal you run `llmfit recommend`; here `subprocess.run` runs the same command from Python and captures what it prints,
and `json.loads` turns that JSON into lists and dicts. We keep the columns that map to our requirements.

</div>

In [8]:
def llmfit(*args):
    """Run `llmfit <args> --json` and return the list of model records."""
    result = subprocess.run(
        ["llmfit", *args, "--json"], capture_output=True, text=True, timeout=180
    )
    return json.loads(result.stdout)["models"]


COLUMNS = [
    "name",
    "params_b",
    "license",
    "memory_required_gb",
    "fit_label",
    "estimated_tps",
    "run_mode",
    "effective_context_length",
]

recommended = pd.DataFrame(llmfit("recommend"))[COLUMNS]
recommended.head(5)

,name,params_b,license,memory_required_gb,fit_label,estimated_tps,run_mode,effective_context_length
0,Qwen/Qwen2.5-Coder-7B-Instruct-GPTQ-Int4,7.62,apache-2.0,4.75,Good,38.1,GPU,8192
1,Qwen/Qwen2.5-Coder-7B-Instruct-AWQ,7.62,apache-2.0,4.75,Good,38.1,GPU,8192
2,Qwen/CodeQwen1.5-7B-AWQ,7.25,tongyi-qianwen-research,4.63,Good,40.1,GPU,8192
3,Qwen/Qwen2.5-7B-Instruct-AWQ,7.62,apache-2.0,4.75,Good,38.1,GPU,8192
4,Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4,7.62,apache-2.0,4.75,Good,38.1,GPU,8192


<div style="text-align: justify; max-width: 90ch">

Useful, and also a reminder that a catalog is not a requirements table: the top rows are whatever scores best on *this* hardware, including variants nobody should ship to customers.
[`abliterated`](https://huggingface.co/blog/mlabonne/abliteration), in the first row's name, means someone removed the model's refusals on purpose.

A catalog can also be wrong. The same row's name says `27B` and its `params_b` says 7.84. By section 4's arithmetic, 27 billion weights at 4 bits need well over 10 GB, not the 4.48 GB listed.
Treat a tool's numbers as data: check a surprising one against the files on the Hub.

</div>

<div style="text-align: justify; max-width: 90ch">

The question we need answered is narrower: do our two tier candidates fit? `llmfit plan` answers it for the file Ollama actually ships (`Q4_K_M`,
section 4) at the 8 000-token context of the requirements table, with a speed estimate for each way to run it:

- **GPU**: the whole model in VRAM, the fastest.
- **CPU offload**: split between VRAM and RAM when the GPU is too small; the speed falls between the other two.
- **CPU only**: no GPU at all; what a laptop without a suitable GPU gets.

</div>

In [9]:
def llmfit_plan(model):
    """Run `llmfit plan` for Ollama's Q4_K_M build and return one row per model."""
    result = subprocess.run(
        [
            "llmfit",
            "plan",
            model,
            "--quant",
            "Q4_K_M",
            "--context",
            "8192",
            "--json",
        ],
        capture_output=True,
        text=True,
        timeout=180,
    )
    plan = json.loads(result.stdout)
    speeds = {path["path"]: path["estimated_tps"] for path in plan["run_paths"]}
    return {
        "name": plan["model_name"],
        "min_vram_gb": plan["minimum"]["vram_gb"],
        "min_ram_gb": plan["minimum"]["ram_gb"],
        "gpu_tps": speeds["gpu"],
        "cpu_only_tps": speeds["cpu_only"],
        "run_mode_here": plan["current"]["run_mode"],
    }


tiers = pd.DataFrame([llmfit_plan("Qwen/Qwen3-1.7B"), llmfit_plan("Qwen/Qwen3-8B")])
tiers.round(1)

,name,min_vram_gb,min_ram_gb,gpu_tps,cpu_only_tps,run_mode_here
0,Qwen/Qwen3-1.7B,2.6,8.0,124.5,11.9,Gpu
1,Qwen/Qwen3-8B,6.4,8.0,30.9,2.9,CpuOffload


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Compare the two rows with your neighbours: `cpu_only_tps` of the 8B row is a speed nobody would call interactive, and `min_vram_gb` is the GPU memory the `gpu` tier needs.
`run_mode_here` is how llmfit would run each model on *this* laptop. That is the whole `TIER` decision, made from a measurement.
Notice what llmfit does **not** know: your languages, your privacy rules, your task. It answers "can I run it?", the requirements table answers "should I?".

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Pull a candidate and compare

Two candidates meet the requirements on paper, that is from cards and estimates: `qwen3:1.7b` and its smaller sibling `qwen3:0.6b`, which would be faster on a weak laptop.
Both are in [Ollama's library](https://ollama.com/library/qwen3), named `family:tag`, where the tag is usually the size:

```bash
ollama pull qwen3:0.6b      # about 520 MB, one-time
```

</div>

In [10]:
SMALL_MODEL = "qwen3:0.6b"
check_ollama(OLLAMA_HOST, [CHAT_MODEL, SMALL_MODEL])

client = ollama.Client(host=OLLAMA_HOST)

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b, qwen3:0.6b


<div style="text-align: justify; max-width: 90ch">

Same three prompts to both models: an English fact from the course, a Greek question, and a small **reasoning trap**, a question whose surface points to the wrong answer (11 is larger than 9,
so 9.11 *looks* larger than 9.9). Thinking stays off, temperature stays at 0, so the only difference is the model. `eval_count` divided by `eval_duration` is the tokens-per-second figure from `lec_01a`.

</div>

In [11]:
prompts = {
    "english": "In one sentence, what does train_test_split do in scikit-learn?",
    "greek": "Σε μία πρόταση, τι κάνει η συνάρτηση train_test_split του scikit-learn; Απάντησε στα ελληνικά.",
    "reasoning": "Which number is larger, 9.11 or 9.9? Answer with the number only.",
}


def compare(models, prompts):
    """Ask every prompt to every model; return one row per answer with its speed."""
    rows = []
    for model in models:
        for name, prompt in prompts.items():
            response = client.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                think=False,
                options={"temperature": 0, "num_predict": 80},
            )
            rows.append(
                {
                    "model": model,
                    "prompt": name,
                    "tok_per_s": round(
                        response.eval_count / (response.eval_duration / 1e9), 1
                    ),
                    "answer": response.message.content.strip().replace("\n", " ")[:110],
                }
            )
    return pd.DataFrame(rows)


compare([SMALL_MODEL, CHAT_MODEL], prompts)

,model,prompt,tok_per_s,answer
0,qwen3:0.6b,english,226.5,Train_test_split() is a function in scikit-lea...
1,qwen3:0.6b,greek,280.4,Συνάρτηση `train_test_split` του scikit-learn ...
2,qwen3:0.6b,reasoning,290.3,The number **9.9** is larger than **9.11**.
3,qwen3:1.7b,english,153.4,`train_test_split` in scikit-learn splits a da...
4,qwen3:1.7b,greek,147.9,Η συνάρτηση `train_test_split` του scikit-lear...
5,qwen3:1.7b,reasoning,202.2,9.9


<div style="text-align: justify; max-width: 90ch">

Read the table against the requirements, not against your taste:

- **Latency**: the small model is about twice as fast; its file is less than half the size, and speed follows memory (section 4).
  These speeds come from one example run with a GPU; yours differ, the order does not.
- **Languages**: the `answer` column stops at 110 characters. In one example run, the full Greek answer of `qwen3:0.6b` only repeated the question;
  `qwen3:1.7b` answered it.
- **Task**: both models chose 9.9, but only `qwen3:1.7b` obeyed "Answer with the number only". A program that reads the answer (`lec_01c`) needs that obedience.

A cheaper model that fails a requirement is not cheaper: it is unusable.

</div>

<div style="text-align: justify; max-width: 90ch">

### 6b. Straight from the Hub

⏱ **Skip if running long.**

Ollama's library is curated; the Hub has everything. Ollama can pull a GGUF file directly from a Hub repository, named `hf.co/<owner>/<repository>:<quantization>`:
section 2's repo id plus a quantization from section 4 ([Hub docs](https://huggingface.co/docs/hub/ollama)). Without the quantization, Ollama takes `Q4_K_M` when the repository has it.

```bash
ollama pull hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0      # about 640 MB
```

The catch: a Hub file brings its own **chat template**, the recipe that turns your messages into the model's input, and Ollama's `think` switch only works with templates it knows.
Watch what the same prompt returns.

</div>

In [12]:
HUB_MODEL = "hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0"
pulled = {m.model for m in client.list().models}

if HUB_MODEL in pulled:
    response = client.chat(
        model=HUB_MODEL,
        messages=[{"role": "user", "content": prompts["reasoning"]}],
        think=False,
        options={"temperature": 0, "num_predict": 60},
    )
    print(response.message.content[:200])
else:
    print(f"Not pulled; run: ollama pull {HUB_MODEL}")

Not pulled; run: ollama pull hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0


<div style="text-align: justify; max-width: 90ch">

The model reasons inside `<think>` tags *in the answer text*, and `think=False` changed nothing: the template did not expose the switch.
Same weights, different packaging, different behaviour. For the course we stay with the library build; the Hub route is there when the library does not have the model you need, and the requirements table gets one more row:
**"does the packaging work with my tooling?"**

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. What thinking costs

⏱ **Skip if running long.**

A reasoning model (`lec_01a`) can write out intermediate steps before its answer, like working a problem on scrap paper:
this is **thinking**. Qwen3's card (section 3) recommends it for complex reasoning, mathematics and coding, and the non-thinking mode for general conversation.
Thinking is generated as tokens, so it costs time in proportion, and on a hosted model you pay for those tokens too (`lec_01a`).
With `think=True`, Ollama returns it apart from the answer, in `message.thinking` ([docs](https://docs.ollama.com/capabilities/thinking)).
Try the trap again, with thinking on.

</div>

In [13]:
started = time.time()
response = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": prompts["reasoning"]}],
    think=True,
    options={"temperature": 0, "num_predict": 1500},
)
elapsed = time.time() - started

print(f"{elapsed:.1f} s, {response.eval_count} tokens generated")
print("thinking (first 300 chars):", (response.message.thinking or "")[:300])
print("answer:", response.message.content.strip())

5.7 s, 812 tokens generated
thinking (first 300 chars): Okay, let's see. The question is asking which number is larger between 9.11 and 9.9. Hmm. Both numbers are decimals, right? So, I need to compare them step by step.

First, let me write them down again to visualize better: 9.11 and 9.9. Both have the same integer part, which is 9. So, the whole numb
answer: To determine which number is larger between 9.11 and 9.9, we compare their decimal parts:

- 9.11 has a decimal part of 0.11.
- 9.9 has a decimal part of 0.9.

Since 0.9 is greater than 0.11, the number 9.9 is larger than 9.11.

---

**Final Answer:**  
$$
\boxed{9.9}
$$


<div style="text-align: justify; max-width: 90ch">

Section 6 got `9.9` from the same model without thinking, in a handful of tokens. Here it spent 930 tokens and 9.5 seconds in one example run with a GPU;
at llmfit's CPU-only estimate of 11.9 tokens/s (section 5), the same tokens would take over a minute. The answer also got worse at its job:
it ignored "Answer with the number only" and wrote a worked solution.

The card adds a warning: in thinking mode use temperature 0.6, not 0, because always picking the most likely token can make the thinking repeat itself without end.
`num_predict=1500` caps that risk here.

Reasoning is a capability you switch on where a decision is worth the wait (lecture 3's agents) and off everywhere else.
That is also a requirements question: "how long may an answer take?"

</div>

<div style="text-align: justify; max-width: 90ch">

## 8. The decision record

A **decision record** is a short note, kept next to the code, that captures one decision and the reasons for it; software teams call it an [architecture decision record](https://adr.github.io/) (ADR).
It exists for the person who comes later: without the reasons, they can only [accept the decision blindly or change it blindly](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions).
Its readers:

- **A colleague or reviewer**, checking that the choice meets the requirements.
- **Whoever signs off licenses and privacy** before the product ships.
- **You in six months**, when a newer model appears and you must decide whether to switch.

In the exercises you write one for a different scenario; in the final assignment you write one for your own.

</div>

<div style="text-align: justify; max-width: 90ch">

A few lines are enough. The record for this course's own choice:

- **Decision:** `qwen3:1.7b` (Q4_K_M) for the `cpu` tier, `qwen3:8b` for the `gpu` tier.
- **Requirements:**
  - RAG over English notes with Greek questions
  - Apache-2.0 or similar
  - 8k+ context
  - 10+ tok/s on a CPU laptop
  - local only
  - one family for both tiers
- **Candidates checked:**
  - Llama 3.2 (no Greek, custom license)
  - Gemma 3 (custom terms)
  - Gemma 4 (Apache-2.0, but its smallest Ollama build is 7.2 GB)
  - Qwen3.5 (Apache-2.0, 201 languages; the successor to test next edition)
  - Qwen3-0.6B (fast, but failed the Greek prompt and the answer format in section 6)
  - Qwen3-1.7B (meets all)
  - Qwen3-8B (GPU only)
- **Evidence:**
  - model cards (section 3)
  - Greek and reasoning prompts (section 6)
  - llmfit fit (section 5)
  - the candidate table in guide `01b_llmfit` (`llmfit plan` at Ollama's `Q4_K_M`)
- **Revisit when:** a newer family in the same license class appears, or a requirement changes (for example Greek notes).

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- Write the **requirements table** first: task, languages, license, context, latency, hardware, privacy, maintenance.
- The **Hugging Face Hub** is GitHub for models: repo ids are `owner/name`; `huggingface_hub` searches it from Python.
- A **model card** has metadata, which search relies on and which can be empty, and text, whose claims you check yourself.
- **Open weights** means downloadable, not free for any use: the license decides.
- **Quantization** trades bytes for a little quality; `Q4_K_M` is the usual default. Memory is parameters × bytes, plus the context window.
- **llmfit** answers "can I run it?"; the table answers "should I?".
- Ollama pulls models from its library or straight from the Hub (`hf.co/...`); compare candidates on *your* prompts.
- Reasoning costs tokens, so time; switch it on only where a decision is worth it.
- Keep a **decision record**.

**Next: `lec_01c`**, prompts as code. The model is chosen; now we make what we send it reproducible.

</div>